In [1]:
# !pip install scikit-optimize

### Logistic Regression

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [3]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
X = train_df
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 함수
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)

shape: (3010, 15)
   NASDAQ_return(%)  Brent Crude Oil_return(%)  Gold Spot_return(%)  \
0          0.593612                   0.545323             0.488488   
1          0.259590                   0.545323             0.696235   
2          0.758918                   0.545323             0.535190   
3          0.592042                   0.545323             0.630347   
4          0.610832                   0.545323             0.657784   

   KOSPI 200 Volume  KOSPI 200 lagged_1_return(%)  KOSPI 200_BB_width  \
0          0.000817                      0.593898            0.385159   
1          0.000552                      0.548072            0.385932   
2          0.000633                      0.616220            0.325038   
3          0.000972                      0.551119            0.257226   
4          0.000917                      0.688352            0.220332   

   KOSPI 200_OG  KOSPI 200_PPO_Hist  Signal4_Buy  Signal4_Sell  \
0      0.728036            0.691619            0  

### 2. Valid 기반 파라미터 최적화

In [4]:
# =========================
# 2. Valid 기반 LR 하이퍼파라미터 상세 탐색
# =========================
import numpy as np
import pandas as pd
import sklearn
from packaging import version

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    log_loss, roc_auc_score, average_precision_score
)

# -------------------------
# 1) 데이터 로드
# -------------------------
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

target_col = "Risk_Label"

X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int)

# valid/test에서 컬럼 불일치가 있으면 0으로 채우지 말고 오류 내는 게 안전함
missing_in_valid = set(X_train_raw.columns) - set(X_valid_raw.columns)
extra_in_valid = set(X_valid_raw.columns) - set(X_train_raw.columns)

if missing_in_valid or extra_in_valid:
    raise ValueError(
        f"train-valid 컬럼 불일치\n"
        f"valid에 없는 train 컬럼: {missing_in_valid}\n"
        f"valid에만 있는 컬럼: {extra_in_valid}"
    )

X_valid_raw = X_valid_raw[X_train_raw.columns]

X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values
feature_names = X_train_raw.columns

print("Train shape:", X_train_model.shape)
print("Valid shape:", X_valid_model.shape)
print("\nTrain class distribution")
print(y_train.value_counts(normalize=True))
print("\nValid class distribution")
print(y_valid.value_counts(normalize=True))


# -------------------------
# 2) 평가 함수
# -------------------------
def get_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(rec * specificity)

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    return {
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": specificity,
        "f1": f1,
        "gmean": gmean,
        "log_loss": ll,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# -------------------------
# 3) sklearn 버전 호환 LR 생성 함수
# -------------------------
def make_lr(C, l1_ratio, max_iter=10000, random_state=42):
    kwargs = dict(
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        random_state=random_state,
        tol=1e-4
    )

    # sklearn 1.8부터 penalty 경고가 뜨므로 버전별 처리
    if version.parse(sklearn.__version__) < version.parse("1.8"):
        kwargs["penalty"] = "elasticnet"

    return LogisticRegression(**kwargs)


# -------------------------
# 4) 1차 coarse grid search
# -------------------------
C_grid_1 = np.logspace(-4, 4, 33)
l1_grid_1 = np.round(np.linspace(0.0, 1.0, 21), 2)

search_history = []

print(f"\n[1차 탐색] 조합 수: {len(C_grid_1) * len(l1_grid_1)}")

for C_val in C_grid_1:
    for l1_val in l1_grid_1:
        model_trial = make_lr(C=C_val, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        m = get_binary_metrics(y_valid, valid_prob, threshold=0.5)

        search_history.append({
            "stage": "coarse",
            "C": C_val,
            "l1_ratio": l1_val,
            "valid_accuracy": m["accuracy"],
            "valid_precision": m["precision"],
            "valid_recall": m["recall"],
            "valid_specificity": m["specificity"],
            "valid_f1": m["f1"],
            "valid_gmean": m["gmean"],
            "valid_log_loss": m["log_loss"],
            "valid_roc_auc": m["roc_auc"],
            "valid_pr_auc": m["pr_auc"],
            "tn": m["tn"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tp": m["tp"]
        })

search_df_1 = pd.DataFrame(search_history)

# 선택 기준:
# 1순위 G-Mean 최대
# 2순위 F1 최대
# 3순위 log_loss 최소
search_df_1 = search_df_1.sort_values(
    by=["valid_gmean", "valid_f1", "valid_log_loss"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("\n[1차 탐색 상위 10개]")
print(search_df_1.head(10))

best_1 = search_df_1.iloc[0]
best_C_1 = best_1["C"]
best_l1_1 = best_1["l1_ratio"]

print("\n[1차 Best]")
print(f"C={best_C_1:.6f}, l1_ratio={best_l1_1:.3f}, "
      f"G-Mean={best_1['valid_gmean']:.4f}, F1={best_1['valid_f1']:.4f}, "
      f"LogLoss={best_1['valid_log_loss']:.4f}")


# -------------------------
# 5) 2차 local refinement
# -------------------------
C_grid_2 = best_C_1 * np.logspace(-0.7, 0.7, 25)
l1_min = max(0.0, best_l1_1 - 0.20)
l1_max = min(1.0, best_l1_1 + 0.20)
l1_grid_2 = np.round(np.linspace(l1_min, l1_max, 25), 3)

search_history_2 = []

print(f"\n[2차 세부 탐색] 조합 수: {len(C_grid_2) * len(l1_grid_2)}")

for C_val in C_grid_2:
    for l1_val in l1_grid_2:
        model_trial = make_lr(C=C_val, l1_ratio=l1_val)
        model_trial.fit(X_train_model, y_train)

        valid_prob = model_trial.predict_proba(X_valid_model)[:, 1]
        m = get_binary_metrics(y_valid, valid_prob, threshold=0.5)

        search_history_2.append({
            "stage": "refine",
            "C": C_val,
            "l1_ratio": l1_val,
            "valid_accuracy": m["accuracy"],
            "valid_precision": m["precision"],
            "valid_recall": m["recall"],
            "valid_specificity": m["specificity"],
            "valid_f1": m["f1"],
            "valid_gmean": m["gmean"],
            "valid_log_loss": m["log_loss"],
            "valid_roc_auc": m["roc_auc"],
            "valid_pr_auc": m["pr_auc"],
            "tn": m["tn"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tp": m["tp"]
        })

search_df_2 = pd.DataFrame(search_history_2)

search_df = pd.concat([search_df_1, search_df_2], ignore_index=True)

search_df = search_df.sort_values(
    by=["valid_gmean", "valid_f1", "valid_log_loss"],
    ascending=[False, False, True]
).reset_index(drop=True)

best_row = search_df.iloc[0]
best_lr_params = {
    "C": float(best_row["C"]),
    "l1_ratio": float(best_row["l1_ratio"])
}

print("\n[최종 Best Hyperparameters]")
print(f"C        : {best_lr_params['C']:.8f}")
print(f"l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"G-Mean   : {best_row['valid_gmean']:.4f}")
print(f"F1       : {best_row['valid_f1']:.4f}")
print(f"Recall   : {best_row['valid_recall']:.4f}")
print(f"Precision: {best_row['valid_precision']:.4f}")
print(f"LogLoss  : {best_row['valid_log_loss']:.4f}")

print("\n[전체 탐색 상위 15개]")
print(search_df.head(15))

# 기록 저장
search_df.to_csv("../../results/lr_hyperparameter_search_history.csv", index=False)


# -------------------------
# 6) Best C/l1_ratio로 다시 학습 후 cutoff 탐색
# -------------------------
best_model_for_cutoff = make_lr(
    C=best_lr_params["C"],
    l1_ratio=best_lr_params["l1_ratio"]
)
best_model_for_cutoff.fit(X_train_model, y_train)

valid_prob = best_model_for_cutoff.predict_proba(X_valid_model)[:, 1]

# 단순 0.01 간격 + valid 확률 분위수 기반 cutoff를 같이 사용
threshold_grid = np.unique(np.r_[
    np.arange(0.01, 0.991, 0.01),
    np.quantile(valid_prob, np.linspace(0.01, 0.99, 99))
])

threshold_history = []

for thr in threshold_grid:
    m = get_binary_metrics(y_valid, valid_prob, threshold=thr)
    threshold_history.append({
        "threshold": float(thr),
        "valid_accuracy": m["accuracy"],
        "valid_precision": m["precision"],
        "valid_recall": m["recall"],
        "valid_specificity": m["specificity"],
        "valid_f1": m["f1"],
        "valid_gmean": m["gmean"],
        "tn": m["tn"],
        "fp": m["fp"],
        "fn": m["fn"],
        "tp": m["tp"]
    })

threshold_df = pd.DataFrame(threshold_history)

# cutoff 선택 기준:
# 1순위 G-Mean 최대
# 2순위 F1 최대
# 3순위 Recall 최대
threshold_df = threshold_df.sort_values(
    by=["valid_gmean", "valid_f1", "valid_recall"],
    ascending=[False, False, False]
).reset_index(drop=True)

best_cutoff = float(threshold_df.loc[0, "threshold"])

print("\n[Valid 기반 최적 Cutoff]")
print(f"Best cutoff : {best_cutoff:.4f}")
print(threshold_df.head(15))

threshold_df.to_csv("../../results/lr_threshold_search_history.csv", index=False)


# -------------------------
# 7) 최종 모델 학습
# -------------------------
model = make_lr(
    C=best_lr_params["C"],
    l1_ratio=best_lr_params["l1_ratio"]
)
model.fit(X_train_model, y_train)

print("\n최종 LR 모델 학습 완료")
print(f"Final C        : {best_lr_params['C']:.8f}")
print(f"Final l1_ratio : {best_lr_params['l1_ratio']:.4f}")
print(f"Final cutoff   : {best_cutoff:.4f}")

Train shape: (3010, 14)
Valid shape: (1232, 14)

Train class distribution
Risk_Label
0    0.617608
1    0.382392
Name: proportion, dtype: float64

Valid class distribution
Risk_Label
0    0.868506
1    0.131494
Name: proportion, dtype: float64

[1차 탐색] 조합 수: 693

[1차 탐색 상위 10개]
    stage          C  l1_ratio  valid_accuracy  valid_precision  valid_recall  \
0  coarse  10.000000      0.10        0.783279         0.294118      0.462963   
1  coarse  10.000000      0.15        0.783279         0.294118      0.462963   
2  coarse   5.623413      0.70        0.783279         0.294118      0.462963   
3  coarse   5.623413      0.75        0.783279         0.294118      0.462963   
4  coarse   5.623413      0.80        0.783279         0.294118      0.462963   
5  coarse  31.622777      0.25        0.783279         0.294118      0.462963   
6  coarse  31.622777      0.30        0.783279         0.294118      0.462963   
7  coarse  31.622777      0.35        0.783279         0.294118      0.46

### 4. 예측 및 성능평가 (train data에서)

In [5]:
# 학습(train) 데이터 기준 확률 계산 후 최적 cutoff 적용
cutoff_to_use = best_cutoff if "best_cutoff" in globals() else 0.5

y_prob = model.predict_proba(X_train_model)[:, 1]
y_pred = (y_prob >= cutoff_to_use).astype(int)

acc = accuracy_score(y_train, y_pred)
prec = precision_score(y_train, y_pred, zero_division=0)
rec = recall_score(y_train, y_pred, zero_division=0)
f1 = f1_score(y_train, y_pred, zero_division=0)
gmean = gmean_score(y_train, y_pred)

print("\n[TRAIN performance]")
print(f"Cutoff   : {cutoff_to_use:.2f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"G-Mean   : {gmean:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_train, y_pred, labels=[0, 1]))

print("\nClassification Report")
print(classification_report(y_train, y_pred, digits=4, zero_division=0))


[TRAIN performance]
Cutoff   : 0.33
Accuracy : 0.6864
Precision: 0.5667
Recall   : 0.7637
F1-score : 0.6506
G-Mean   : 0.6983

Confusion Matrix
[[1187  672]
 [ 272  879]]

Classification Report
              precision    recall  f1-score   support

           0     0.8136    0.6385    0.7155      1859
           1     0.5667    0.7637    0.6506      1151

    accuracy                         0.6864      3010
   macro avg     0.6902    0.7011    0.6831      3010
weighted avg     0.7192    0.6864    0.6907      3010



### 5. test data 성능 평가

In [6]:
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

X_test_df = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int)

missing_in_test = set(feature_names) - set(X_test_df.columns)
extra_in_test = set(X_test_df.columns) - set(feature_names)

if missing_in_test or extra_in_test:
    raise ValueError(
        f"train-test 컬럼 불일치\n"
        f"test에 없는 train 컬럼: {missing_in_test}\n"
        f"test에만 있는 컬럼: {extra_in_test}"
    )

X_test_df = X_test_df[feature_names]
X_test = X_test_df.values

y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_cutoff).astype(int)

test_metrics = get_binary_metrics(y_test, y_test_prob, threshold=best_cutoff)

print("\n[TEST performance]")
print(f"Cutoff    : {best_cutoff:.4f}")
print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
print(f"Specificity: {test_metrics['specificity']:.4f}")
print(f"F1-score  : {test_metrics['f1']:.4f}")
print(f"G-Mean    : {test_metrics['gmean']:.4f}")
print(f"ROC-AUC   : {test_metrics['roc_auc']:.4f}")
print(f"PR-AUC    : {test_metrics['pr_auc']:.4f}")
print(f"LogLoss   : {test_metrics['log_loss']:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))

print("\nClassification Report")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))


[TEST performance]
Cutoff    : 0.3307
Accuracy  : 0.6265
Precision : 0.1920
Recall    : 0.7283
Specificity: 0.6137
F1-score  : 0.3039
G-Mean    : 0.6685
ROC-AUC   : 0.7259
PR-AUC    : 0.2873
LogLoss   : 0.4927

Confusion Matrix
[[448 282]
 [ 25  67]]

Classification Report
              precision    recall  f1-score   support

           0     0.9471    0.6137    0.7448       730
           1     0.1920    0.7283    0.3039        92

    accuracy                         0.6265       822
   macro avg     0.5696    0.6710    0.5243       822
weighted avg     0.8626    0.6265    0.6955       822

